## Libraries

In [120]:
import pandas as pd
import numpy as np
import requests

## Loading Data

Retrieving Dataset from EIA api

In [144]:
API_KEY = "rGIsttjuD3E6wa6wTqZwsgegfhWfRbE6CEMb6NiR"
url = "https://api.eia.gov/v2/electricity/rto/daily-region-data/data/"

TYPES = ["Demand", "Day-ahead demand forecast"]  

params = {
    "api_key": API_KEY,
    "frequency": "daily",
    "data[0]": "value",
    "facets[respondent][]": "PJM",
    "facets[type-name][]": TYPES,
    "start": "2022-01-01",
    "end": "2024-12-31",
    "length": 5000
}

r = requests.get(url, params=params)
data = r.json()

if "response" not in data or "data" not in data["response"]:
    raise ValueError("API returned no data. Check your API key and filters.")

df = pd.DataFrame(data["response"]["data"])

df.to_csv("data/electric.csv")


ValueError: API returned no data. Check your API key and filters.

In [132]:
df_electric = pd.read_csv("data/electric.csv")
print(df_electric.shape)
display(df_electric.head(3))

(5000, 10)


,Unnamed: 0,period,respondent,respondent-name,type,type-name,timezone,timezone-description,value,value-units
0,0,2024-12-31,PJM,"PJM Interconnection, LLC",D,Demand,Arizona,Arizona,2112461,megawatthours
1,1,2024-12-31,PJM,"PJM Interconnection, LLC",D,Demand,Central,Central,2115276,megawatthours
2,2,2024-12-31,PJM,"PJM Interconnection, LLC",D,Demand,Eastern,Eastern,2119871,megawatthours


In [123]:
df_noaa = pd.read_csv("data/noaa.csv", low_memory=False)
print(df_noaa.shape)
display(df_noaa.head(3))

(318144, 8)


,STATION,NAME,DATE,AWND,PRCP,TAVG,TMAX,TMIN
0,US1NJHN0019,"LEBANON 3.2 SW, NJ US",2022-01-01,NaN,0.05,NaN,NaN,NaN
1,US1NJHN0019,"LEBANON 3.2 SW, NJ US",2022-01-02,NaN,0.62,NaN,NaN,NaN
2,US1NJHN0019,"LEBANON 3.2 SW, NJ US",2022-01-03,NaN,0.00,NaN,NaN,NaN


## Data Cleaning

### Df_electric

In [124]:
# Rename period column to date

df_electric["DATE"] = pd.to_datetime(df_electric["period"])  
df_electric["DATE"] = df_electric["DATE"].dt.date  

In [125]:
full_range = pd.date_range(start="2022-01-01", end="2024-12-31", freq="D").date
missing_dates = sorted(set(full_range) - set(df_electric["DATE"]))

print(f"Missing dates: {len(missing_dates)}")
print(missing_dates)

Missing dates: 845
[datetime.date(2022, 1, 1), datetime.date(2022, 1, 2), datetime.date(2022, 1, 3), datetime.date(2022, 1, 4), datetime.date(2022, 1, 5), datetime.date(2022, 1, 6), datetime.date(2022, 1, 7), datetime.date(2022, 1, 8), datetime.date(2022, 1, 9), datetime.date(2022, 1, 10), datetime.date(2022, 1, 11), datetime.date(2022, 1, 12), datetime.date(2022, 1, 13), datetime.date(2022, 1, 14), datetime.date(2022, 1, 15), datetime.date(2022, 1, 16), datetime.date(2022, 1, 17), datetime.date(2022, 1, 18), datetime.date(2022, 1, 19), datetime.date(2022, 1, 20), datetime.date(2022, 1, 21), datetime.date(2022, 1, 22), datetime.date(2022, 1, 23), datetime.date(2022, 1, 24), datetime.date(2022, 1, 25), datetime.date(2022, 1, 26), datetime.date(2022, 1, 27), datetime.date(2022, 1, 28), datetime.date(2022, 1, 29), datetime.date(2022, 1, 30), datetime.date(2022, 1, 31), datetime.date(2022, 2, 1), datetime.date(2022, 2, 2), datetime.date(2022, 2, 3), datetime.date(2022, 2, 4), datetime.date

In [126]:
missing_df = pd.DataFrame({"DATE": missing_dates})
df_electric = pd.concat([df_electric, missing_df], ignore_index=True)
df_electric = df_electric.sort_values("DATE").reset_index(drop=True)

### Df_noaa

In [127]:
# Convert DATE column to datetime
df_noaa["DATE"] = pd.to_datetime(df_noaa["DATE"])

# Need only daily
df_noaa["DATE"] = df_noaa["DATE"].dt.date

In [129]:
# Check for missing dates / duplicates

full_range = pd.date_range(start="2022-01-01", end="2024-12-31", freq="D")

missing_dates = full_range.difference(df_noaa["DATE"])

print("Number of missing rows:", len(missing_dates))
print(missing_dates)

# check for duplicates
duplicates = df_noaa[df_noaa.duplicated(subset=["DATE"], keep=False)]
print("Duplicate rows:", len(duplicates))

Number of missing rows: 0
DatetimeIndex([], dtype='datetime64[ns]', freq='D')
Duplicate rows: 318144


In [ ]:
# Dropping Duplicates
df_noaa = df_noaa.drop_duplicates(subset=["DATE"])

In [ ]:
# Fill missing dates with Nan

missing_df = pd.DataFrame({"DATE": missing_dates})
df_noaa = pd.concat([df_noaa, missing_df], ignore_index=True)

df_noaa = df_noaa.sort_values("DATE").reset_index(drop=True)

### Data Merging

In [ ]:
df_noaa["DATE"] = pd.to_datetime(df_noaa["DATE"]).dt.date  
df_electric["DATE"] = pd.to_datetime(df_electric["DATE"]).dt.date  

df_merged = pd.merge(df_electric, df_noaa, on="DATE", how="left")

In [ ]:
df_merged.to_csv("data/pjm_noaa_merged.csv", index=False)